# LangChain 模型调用: `init_chat_model`

`init_chat_model()` 是 LangChain 中最常用的函数之一，让你用统一的方式连接 20 多种模型提供商，不需要记忆每个提供商的类名和参数差异。
当然，你可能需要安装对应的包，比如 deepseek 需要 langchain-deepseek。

## 基本用法

In [1]:
from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnableConfig

from models import Models
from settings import settings

model = init_chat_model(
    # 需要安装：langchain-deepseek
    "deepseek:deepseek-v4-flash",                 # 模型名称，推荐 "提供商:模型名" 格式
    model_provider=None,   # 单独指定提供商（可选）
    temperature=0.7,       # 控制随机性, 0~2, 值越小输出越稳定
    max_tokens=200,        # 限制输出长度
    timeout=30,            # 请求超时（秒）
    max_retries=2,         # 失败重试次数
    api_key=settings.deepseek_api_key
)
model

ChatDeepSeek(metadata={'lc_versions': {'langchain-core': '1.5.6', 'langchain': '1.3.15', 'langchain-openai': '1.5.2'}}, profile={'name': 'DeepSeek V4 Flash', 'release_date': '2026-04-24', 'last_updated': '2026-04-24', 'open_weights': True, 'max_input_tokens': 1000000, 'max_output_tokens': 384000, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001C8A36B8050>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001C8A36B8C20>, root_client=<openai.OpenAI object at 0x000001C8A27A7CB0>, root_async_client=<openai.AsyncOpenAI object at 0x000001C8A36B81A0>, model_name='deepseek-v4-flash', temperature=0.7, model_kwargs={}, op

## 指定模型

### 方式1：`provider:model` 格式（推荐方式）

In [2]:
model = init_chat_model("deepseek:deepseek-chat")

### 方式2：自动推断（不一定100%正确）

In [3]:
model = init_chat_model("deepseek-chat")

### 方式3：用 `model_provider` 单独指定

In [4]:
model = init_chat_model("deepseek-chat", model_provider="deepseek")

## 可配置模型

配置时可以不指定模型，运行时动态切换。

In [5]:
configurable_model = init_chat_model(temperature=0.6)


config: RunnableConfig = {
    "configurable": {
        "model": Models.DEEPSEEK_V4_FLASH,
        "api-key": settings.deepseek_api_key
    }
}

config2: RunnableConfig = {
    "configurable": {
        "model": Models.DEEPSEEK_V4_PRO,
        "api-key": settings.deepseek_api_key
    }
}

resp = configurable_model.invoke("一句话，介绍下你自己。", config=config)
resp2 = configurable_model.invoke("一句话,介绍下你自己。", config=config2)
resp, resp2

(AIMessage(content='你好，我是DeepSeek，一个由深度求索公司打造的AI助手，乐于为你解答问题、提供帮助！', additional_kwargs={'refusal': None, 'reasoning_content': '我们只需要用一句话介绍自己。用户要求“一句话，介绍下你自己。”所以回答要简短，一句话。可以介绍为AI助手，功能等。注意不要多余。'}, response_metadata={'token_usage': {'completion_tokens': 62, 'prompt_tokens': 89, 'total_tokens': 151, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 36, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 89}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': '93c7d0f3-863b-481a-8bc5-eb509cc2c2c4', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a01d20-1d1a-79e3-a3cd-364be44fa83f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_toke

# LangChain 模型参数速查手册

| 参数             | 类型      | 作用                                                 | 推荐取值                                                                                          | 典型场景                                    |
|------------------|-----------|------------------------------------------------------|---------------------------------------------------------------------------------------------------|---------------------------------------------|
| `temperature`    | float     | 控制输出随机性（温度）                               | `0.0` ~ `2.0`<br>`0.0~0.3`（稳定）<br>`0.5~0.7`（适中）<br>`0.8~1.2`（多样）<br>`1.3~2.0`（随机） | 精确任务用低值，创意任务用高值              |
| `max_tokens`     | int       | 限制输出最大 Token 数                                | `100` ~ `2000`（根据任务）                                                                        | 控制成本，防止输出过长                      |
| `top_p`          | float     | 核采样（nucleus sampling），从累积概率达标的词中采样 | `0.0` ~ `1.0`，默认 `1.0`                                                                         | 替代 `temperature` 控制随机性（两者选其一） |
| `stop`           | list[str] | 遇到指定序列立即停止生成                             | 如 `["。", "END"]`                                                                                | 精确控制输出结尾格式                        |
| `seed`           | int       | 固定随机种子，使输出可复现                           | 任意整数（如 `42`）                                                                               | 需要结果稳定一致时                          |
| `timeout`        | float     | 单次请求最大等待时间（秒）                           | `30` ~ `60`                                                                                       | 生产环境防止请求卡死                        |
| `max_retries`    | int       | 请求失败后的重试次数                                 | `2` ~ `3`                                                                                         | 应对偶发网络或限流问题                      |
| `base_url`       | str       | 自定义 API 服务地址                                  | URL 字符串（如 `"http://localhost:11434"`）                                                       | 使用代理、中转或本地模型（如 Ollama）       |
| `model`          | str       | 必填，模型标识符                                     | 如 `"deepseek:deepseek-chat"`                                                                     | 指定要调用的模型                            |
| `model_provider` | str       | 指定提供商（可选，可自动推断）                       | `"openai"`, `"anthropic"`, `"deepseek"` 等                                                        | 在 `model` 未含前缀时补充                   |

## 参数详解与注意事项

### `temperature` vs `top_p`
- **只调其中一个**：同时调节可能使输出行为不可预测，通常选 `temperature` 更直观。
- `temperature=0` **不等于绝对确定性**：由于浮点运算，微小差异仍可能存在，如需完全复现可配合 `seed`。

### `max_tokens` 是硬截断
- 设置过短会导致回答在句子中间突然中断（例如 `max_tokens=10` 只输出几个单词）。
- 实际字数约为 `max_tokens × 0.75` 英文单词或 `× 0.5` 中文字符。

### `timeout` 与 `max_retries` 的总耗时
- 若 `timeout=30`, `max_retries=3`，全部失败最多耗时 `(3+1) × 30 = 120` 秒才会抛出异常。

### `base_url` 使用提醒
- 当自定义 `base_url` 时，`model_provider` 决定了请求的消息格式，需确保目标服务（如代理网关）兼容该格式。

### `seed` 可复现性
- 部分模型提供商可能不保证完全复现（如某些开源模型），建议先测试验证。

## 参数示例

### `temperature` 控制模型创造性与确定性
temperature 是最常用的参数，取值范围 0 到 2。它控制模型输出的随机程度。

In [6]:
model_low = init_chat_model(model=Models.DEEPSEEK_V4_FLASH, temperature=0.1, api_key=settings.deepseek_api_key)
model_high = init_chat_model(model=Models.DEEPSEEK_V4_FLASH, temperature=1.2, api_key=settings.deepseek_api_key)

ask = "30个字介绍一下 LangChain"
resp1 = model_low.invoke(ask)
resp2 = model_low.invoke(ask)

resp1.content, resp2.content

('LangChain是一个用于构建大语言模型应用的开源框架，支持链式调用与工具集成。',
 'LangChain 是一个开源框架，用于构建大语言模型应用，支持链式调用和智能体。')

In [7]:
resp3 = model_high.invoke(ask)
resp4 = model_high.invoke(ask)

resp3.content, resp4.content

('LangChain是开源框架，用于构建大语言模型应用，简化组件调用与流程编排。',
 'LangChain是开发大语言模型应用的框架，简化链式调用与记忆管理。')